In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/albertnathaniel12/food-recipes-dataset/Indonesian_Food_Recipes.csv


In [2]:
# Load dataset
df = pd.read_csv('/kaggle/input/datasets/albertnathaniel12/food-recipes-dataset/Indonesian_Food_Recipes.csv')

# Melihat 5 baris pertama
print("--- 5 Baris Pertama ---")
print(df.head())

# Melihat informasi kolom, tipe data, dan missing values
print("\n--- Informasi Struktur Data ---")
df.info()

# Melihat statistik deskriptif (opsional)
print("\n--- Statistik Deskriptif ---")
print(df.describe())

--- 5 Baris Pertama ---
                      Title  \
0          Ayam Woku Manado   
1  Ayam goreng tulang lunak   
2          Ayam cabai kawin   
3               Ayam Geprek   
4               Minyak Ayam   

                                         Ingredients  \
0  1 Ekor Ayam Kampung (potong 12)--2 Buah Jeruk ...   
1  1 kg ayam (dipotong sesuai selera jangan kecil...   
2  1/4 kg ayam--3 buah cabai hijau besar--7 buah ...   
3  250 gr daging ayam (saya pakai fillet)--Secuku...   
4  400 gr kulit ayam & lemaknya--8 siung bawang p...   

                                               Steps  Loves  \
0  1) Cuci bersih ayam dan tiriskan. Lalu peras j...      1   
1  1) Haluskan bumbu2nya (BaPut, ketumbar, kemiri...      1   
2  1) Panaskan minyak di dalam wajan. Setelah min...      2   
3  1) Goreng ayam seperti ayam krispi\n2) Ulek se...     10   
4  1) Cuci bersih kulit ayam. Sisihkan\n2) Ambil ...      4   

                                                 URL Category  \
0  https

In [3]:
!pip install -q --no-deps langchain langchain-community langchain-huggingface faiss-cpu

In [4]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Inisialisasi kembali model embedding
# Ini adalah "mesin" yang mengubah teks resep menjadi angka (vektor)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. Sekarang jalankan ulang pembuatan vectorstore Anda
documents = []
for index, row in df.iterrows():
    content = f"Resep: {row['Title']}\n\nBahan:\n{row['Ingredients']}\n\nLangkah:\n{row['Steps']}"
    metadata = {
        "source": str(row['URL']),
        "title": str(row['Title'])
    }
    doc = Document(page_content=content, metadata=metadata)
    documents.append(doc)

vectorstore = FAISS.from_documents(documents, embeddings)
print("Vector Store berhasil diperbarui dengan Metadata dan Embeddings!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector Store berhasil diperbarui dengan Metadata dan Embeddings!


In [5]:
!pip install -q -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.4 MB/s eta 0:00:00


In [11]:
import os
import time
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. API Key
os.environ["GOOGLE_API_KEY"] = "api key here"

# 2. Inisialisasi LLM dengan logika penanganan error yang kuat
def initialize_llm():
    # Menggunakan model terbaru yang sukses sebelumnya
    candidates = ["gemini-2.5-flash", "models/gemini-2.0-flash", "gemini-3.0-pro", "gemini-pro"]
    
    for model_name in candidates:
        try:
            model = ChatGoogleGenerativeAI(model=model_name, temperature=0.7)
            model.invoke("test")
            print(f"Koneksi Berhasil! Menggunakan model: {model_name}")
            return model
        except Exception:
            print(f"Model {model_name} tidak dapat diakses, mencoba model lain...")
            continue
    return None

llm = initialize_llm()

if llm:
    # 3. Template Prompt yang diperkuat instruksinya
    # Modifikasi Template agar lebih instruktif
    template = """Anda adalah koki ahli Indonesia yang ramah dan inspiratif. 
    Tugas Anda adalah memberikan resep lengkap berdasarkan konteks yang tersedia.
    
    Struktur Jawaban:
    1. Sapaan hangat kepada pengguna.
    2. Daftar Bahan-bahan yang diperlukan (gunakan bullet points).
    3. Tutorial Langkah-langkah Pembuatan (gunakan penomoran 1, 2, 3...).
    4. Tips singkat agar masakan lebih lezat (jika ada).
    5. Link Sumber URL di bagian paling bawah.
    
    Jika resep tidak ditemukan dalam konteks, katakan dengan sopan bahwa Anda belum memilikinya.
    
    Konteks:
    {context}
    
    Pertanyaan: {question}
    Jawaban Koki:"""
    
    prompt = ChatPromptTemplate.from_template(template)
    
    # 4. Fungsi format_docs untuk menggabungkan konten + metadata URL
    def format_docs_with_metadata(docs):
        formatted = []
        for doc in docs:
            # Mengambil konten resep dan sumber dari metadata
            source_url = doc.metadata.get('source', 'Tidak tersedia')
            title = doc.metadata.get('title', 'Resep Tanpa Judul')
            
            content = f"JUDUL: {title}\nRESEP: {doc.page_content}\nSUMBER: {source_url}"
            formatted.append(content)
        return "\n\n---\n\n".join(formatted)

    # 5. Setting Retriever
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # 6. Membangun RAG Chain dengan LCEL
    # Pastikan fungsi format_docs_with_metadata masih menggunakan metadata 'source' dan 'title'
    rag_chain = (
        {"context": retriever | format_docs_with_metadata, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    
    # --- Uji Coba Lagi ---
    try:
        query = "Bagaimana cara membuat ayam geprek?"
        print(f"Menghubungi asisten koki untuk tutorial: {query}\n")
        response = rag_chain.invoke(query)
        print(response)
    except Exception as e:
        print(f"Terjadi kesalahan: {e}")
else:
    print("Gagal inisialisasi LLM. Periksa API Key atau kuota Free Tier Anda.")

Koneksi Berhasil! Menggunakan model: gemini-2.5-flash
Menghubungi asisten koki untuk tutorial: Bagaimana cara membuat ayam geprek?

Halo, sahabat kuliner Indonesia! Senang sekali bisa berbagi resep spesial dengan Anda hari ini. Ayam geprek memang juaranya, ya! Perpaduan ayam renyah dengan sambal pedas yang nendang itu, siapa bisa menolak?

Mari kita siapkan apron dan mulai memasak "Ayam Geprek" yang lezat ini!

### Resep Ayam Geprek

**Bahan-bahan yang diperlukan:**

*   250 gr Ayam bagian Dada
*   200 gr Tepung Ayam instan (misalnya Sajiku Golden Crispy)
*   **Untuk Sambal:**
    *   7 Buah Cabe Rawit (sesuaikan selera pedas Anda)
    *   1/2 siung Bawang Putih
    *   Garam secukupnya

**Tutorial Langkah-langkah Pembuatan:**

1.  **Persiapan Ayam:** Cuci bersih ayam bagian dada, lalu potong sesuai selera Anda. Beri beberapa sayatan pada daging ayam agar bumbu lebih mudah meresap. Setelah itu, lumuri dengan perasan jeruk nipis dan rebus sebentar.
2.  **Adonan Tepung Basah:** Larutkan 

In [7]:
query_test = "Ayam Woku Manado"
docs = vectorstore.similarity_search(query_test, k=3)

for i, doc in enumerate(docs):
    print(f"Hasil ke-{i+1}: {doc.metadata['title']}")

Hasil ke-1: Olahan ayam untuk isian roti/bapao
Hasil ke-2: Ayam Geprek
Hasil ke-3: Ayam balado...
